# E791 $D^+\to\pi^-\pi^+\pi^+$ — Fit 2 generation

Generation example based on Fit 2 of E791, arXiv:hep-ex/0007028v2.

The decay channel is declared once. Standard particle masses are resolved with the `particle` package, while resonance masses/widths that are specific to the E791 model are supplied as explicit overrides. Identical-pion symmetrization is automatic inside each resonance component.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance,
    enable_x64, weighted_resample,
)

enable_x64()


## 1. Channel and Fit 2 coefficients


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
print("m(D+) =", channel.parent_mass, "GeV")
print("daughter masses =", channel.daughter_masses, "GeV")

fit2_polar = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

fit2_xy = {name: polar_to_xy(*value) for name, value in fit2_polar.items()}
for name, (x, y) in fit2_xy.items():
    print(f"{name:8s}: x={x:+.5f}, y={y:+.5f}")


## 2. Declarative amplitude model

Only the channel, resonance pair and analysis-specific resonance parameters are declared. Parent/daughter/bachelor masses and the exchanged identical-pion contribution are inferred automatically.


In [ ]:
def c(name):
    return RealImag(*fit2_xy[name])

components = [
    Resonance("sigma", pair=(0,1), coefficient=c("sigma"), mass=0.478, width=0.324, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", pair=(0,1), coefficient=c("rho770"), mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", pair=(0,1), coefficient=c("f0_980"), mass=0.975, width=0.044, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", pair=(0,1), coefficient=c("f2_1270"), mass=1.275, width=0.185, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", pair=(0,1), coefficient=c("f0_1370"), mass=1.434, width=0.173, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", pair=(0,1), coefficient=c("rho1450"), mass=1.465, width=0.310, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(c("NR")),
]

model = DecayModel(channel, components)
model.amplitude_model.components


## 3. Weighted candidate pool and target density


In [ ]:
N_POOL = 1_000_000
N_TOY = 100_000

pool = model.generate_phase_space(N_POOL, seed=2000)
data_pool = pool.as_dict()
intensity = model.intensity(data_pool)
target_weights = pool.weights * intensity

print("pool:", pool.size)
print("finite target weights:", bool(jnp.all(jnp.isfinite(target_weights))))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
h0 = axes[0].hist2d(np.asarray(pool.s12), np.asarray(pool.s13), bins=120, weights=np.asarray(pool.weights))
fig.colorbar(h0[3], ax=axes[0], label="phase-space weight")
axes[0].set_title("Weighted phase space")
h1 = axes[1].hist2d(np.asarray(pool.s12), np.asarray(pool.s13), bins=120, weights=np.asarray(target_weights))
fig.colorbar(h1[3], ax=axes[1], label=r"$w_{PS}|A|^2$")
axes[1].set_title("E791 Fit 2 model density")
for ax in axes:
    ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
    ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
plt.show()


## 4. Resample 100k unweighted pseudo-data events


In [ ]:
toy = weighted_resample(jax.random.key(791), pool, target_weights, N_TOY, replace=True)
print("toy events:", toy.size)

fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("100k unweighted pseudo-data")
plt.show()


## 5. Symmetrized $\pi^+\pi^-$ projection


In [ ]:
s_pm = np.concatenate([np.asarray(toy.s12), np.asarray(toy.s13)])
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(s_pm, bins=100, histtype="step", linewidth=1.6)
ax.set_xlabel(r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
ax.set_ylabel("Entries / bin")
ax.set_title(r"Fit 2-inspired $s_{12}+s_{13}$ projection")
plt.show()


## 6. Component projections and interference


In [ ]:
built_components = model.amplitude_model.components
bins = np.linspace(float(min(jnp.min(pool.s12), jnp.min(pool.s13))), float(max(jnp.max(pool.s12), jnp.max(pool.s13))), 100)
fig, ax = plt.subplots(figsize=(9, 6))
for component in built_components:
    amp = component.value(data_pool)
    w = pool.weights * jnp.abs(amp)**2
    s = np.concatenate([np.asarray(pool.s12), np.asarray(pool.s13)])
    ww = np.concatenate([np.asarray(w), np.asarray(w)])
    hist, edges = np.histogram(s, bins=bins, weights=ww, density=True)
    ax.step(0.5*(edges[:-1]+edges[1:]), hist, where="mid", label=component.name)
ax.set_xlabel(r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
ax.set_ylabel("Normalized component intensity")
ax.legend(ncol=2)
plt.show()

coherent = jnp.abs(model.amplitude(data_pool))**2
incoherent = sum(jnp.abs(component.value(data_pool))**2 for component in built_components)
interference = pool.weights * (coherent - incoherent)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.asarray(pool.s12), bins=100, weights=np.asarray(interference), histtype="step", label=r"$s_{12}$")
ax.hist(np.asarray(pool.s13), bins=100, weights=np.asarray(interference), histtype="step", label=r"$s_{13}$")
ax.axhline(0.0, linewidth=1)
ax.set_xlabel(r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
ax.set_ylabel("Net interference weight")
ax.legend()
plt.show()
